# Prompt-Based Generation

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
import json
import shutil
from diffusers import AutoPipelineForText2Image
from PIL import Image
from tqdm import tqdm
from pathlib import Path
from IPython.display import display
from transformers import CLIPModel, CLIPProcessor
from skimage.metrics import structural_similarity as ssim


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\victus\Documents\AIML_Sem-5\ML\Project\synthetic-medical-image-generation\env\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\victus\Documents\AIML_Sem-5\ML\Project\synthetic-medical-image-generation\env\Lib\site-packages\traitlets\config\application.py", line 1080, in launch_instance
    app.start()
  File "c:\Users\victus\Documents\AIML_Sem

RuntimeError: Failed to import diffusers.pipelines.auto_pipeline because of the following error (look up to see its traceback):
Failed to import diffusers.pipelines.aura_flow.pipeline_aura_flow because of the following error (look up to see its traceback):
Failed to import diffusers.models.transformers.auraflow_transformer_2d because of the following error (look up to see its traceback):
Failed to import diffusers.loaders.unet because of the following error (look up to see its traceback):
name 'nn' is not defined

## Data Visualization

### General Information

In [ ]:
pd.set_option('display.max_colwidth', None)
data = pd.read_csv("./data/HAM10000_metadata.csv")
data.head()

In [ ]:
print(f'{data.shape[0]} samples before dropping patients NaN values')
data = data.dropna()
data = data.reset_index(drop=True)
print(f'{data.shape[0]} samples after dropping patients NaN values')

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(20,5))

sns.histplot(data=data, bins= 10, x = 'age', kde=True, color=sns.color_palette("crest")[3], ax=axes[0])
axes[0].set_title('Age')

sns.countplot(data=data, x = 'sex', hue='sex', palette='deep', ax=axes[1])
axes[1].set_title("Sex")

sns.countplot(data=data, x = 'dx', palette='deep', hue = 'dx', ax=axes[2])
axes[2].set_title('Diagnoses')

plt.suptitle('HAM10000 Data Distribution', fontweight='bold', fontsize=14)
plt.show()

In [ ]:
plt.figure(figsize=(20,9))
sns.countplot(data=data, x='localization', palette='deep', hue='localization')
plt.title('Localization Area', fontweight='bold',fontsize=14)
plt.show()

### Display Random Image

In [ ]:
index = np.random.randint(0,data.shape[0])

def display_image(index):
    image_id = data['image_id'][index]
    if not image_id.endswith('.jpg'):
        image_id += '.jpg'
    file1 = os.path.join('data', 'HAM10000_images_part_1', image_id)
    file2 = os.path.join('data', 'HAM10000_images_part_2', image_id)
    
    if os.path.exists(file1):
        return file1
    elif os.path.exists(file2):
        return file2
    else:
        raise FileNotFoundError

image = display_image(index)

In [ ]:
def display_metadata(index):
    image_metadata = pd.DataFrame(data.iloc[index,:]).reset_index()
    image_metadata.columns = ['Feature', 'Value']
    return image_metadata

In [ ]:
image = Image.open(image)
plt.imshow(image)
plt.axis('off')
plt.show()
display_metadata(index)


## Prompt Builder

In [ ]:
diagnosis_map = {
    'akiec': 'actinic keratosis / Bowen\'s disease',
    'bcc': 'basal cell carcinoma',
    'bkl': 'benign keratosis-like lesion',
    'df': 'dermatofibroma',
    'mel': 'melanoma',
    'nv': 'melanocytic nevus',
    'vasc': 'vascular lesion'
}

In [ ]:
prompts = {
    "full_metadata": [
        "Dermoscopy image of a {dx} on the {site} of a {age}-year-old {sex}.",
        "Dermoscopic photograph showing a {dx} on the {site} in a {age}-year-old {sex}.",
        "Skin lesion ({dx}) located on the {site} of a {age}-year-old {sex}.",
        "Dermoscopy of a {dx} on the {site} in a {age}-year-old {sex}.",
        "Cutaneous {dx} located on the {site} of a {age}-year-old {sex}, dermoscopy image.",
        "Dermoscopic close-up showing a {dx} on the {site} of a {age}-year-old {sex}.",
        "Pigmented lesion ({dx}) on the {site} in a {age}-year-old {sex}, dermoscopic view."
    ],

    "no_sex": [
        "High-resolution dermoscopy of a {dx} on the {site}.",
        "Clinical dermoscopic view of a {dx} on the {site}.",
        "Dermoscopic image presenting a {dx} on the {site}.",
        "Magnified skin image of a {dx} found on the {site}.",
        "Detailed dermoscopy of a {dx} lesion on the {site}.",
        "Macro dermoscopic photograph depicting a {dx} on the {site}.",
        "Skin surface dermoscopy showing a {dx} on the {site}.",
        "Dermoscopic evaluation image of a {dx} on the {site}.",
        "Standardized dermoscopy photograph of a {dx} on the {site}.",
        "Polarized dermoscopy image of a {dx} on the {site}.",
        "Clinical close-up dermoscopy showing {dx} on the {site}.",
        "{dx} on the {site}, dermoscopy image.",
        "Dermoscopic photo: {dx} on the {site}.",
        "Dermoscopy view of lesion: {dx} on {site}.",
        "Magnified dermoscopy image of {dx} on the {site}."
    ],

    "no_localization": [
        "Diagnostic dermoscopy capture depicting a {dx}.",
        "Dermatology-focused dermoscopic image showing a {dx} lesion.",
        "Dermoscopy of {dx}.",
        "High-quality dermoscopic image of a {dx}.",
        "Clinical dermoscopic capture showing {dx}."
    ],

    "dx_only": [
        "Dermoscopic image of a {dx}.",
        "Skin lesion consistent with {dx}, dermoscopy view.",
        "Close-up dermoscopic capture revealing {dx}.",
        "Dermatology image showing features of {dx}.",
        "Dermoscopy photograph displaying {dx}."
    ]
}

In [ ]:
def create_prompt(df_row):
    if df_row['localization'] == 'unknown' and df_row['sex'] == 'unknown':
        prompt_choice = np.random.choice(prompts['dx_only'])
    elif df_row['localization'] == 'unknown':
        prompt_choice = np.random.choice(prompts['no_localization'])
    elif df_row['sex'] == 'unknown':
        prompt_choice = np.random.choice(prompts['no_sex'])
    else:
        categories = ['full_metadata', 'no_sex', 'no_localization', 'dx_only']
        selected_category = np.random.choice(categories, p=[0.7,0.1,0.1,0.1])
        prompt_choice = np.random.choice(prompts[selected_category])
        
    
    if 'dx' in prompt_choice:
        diagnoses = df_row['dx']
        mapping = diagnosis_map[diagnoses]
        prompt_choice = prompt_choice.replace('{dx}', mapping)
    if 'site' in prompt_choice:
        prompt_choice = prompt_choice.replace('{site}', df_row['localization'])
    if 'age' in prompt_choice:
        prompt_choice = prompt_choice.replace('{age}', str(int(df_row['age'])))
    if 'sex' in prompt_choice:
        prompt_choice = prompt_choice.replace('{sex}', df_row['sex'])
    
    return prompt_choice

In [ ]:
row, col = data.shape
data['Prompt'] = np.full((row), 'NA')

for idx in range(row):
    patient = data.iloc[idx,:]
    prompt = create_prompt(patient)
    data.loc[idx, 'Prompt'] = prompt
    
data.head(10)

In [ ]:
image = display_image(index)
image = Image.open(image)
plt.imshow(image)
plt.axis('off')
display_metadata(index)

## Stable Diffusion

In [ ]:
device = (
    'cuda' if torch.cuda.is_available() else
    'mps' if torch.backends.mps.is_available() else
    'cpu'
)

Creating a unified folder with the desired input structure for stable diffusion fine-tuning

In [ ]:
src1 = Path("data/HAM10000_images_part_1")
src2 = Path("data/HAM10000_images_part_2")
dst  = Path("data/ham_lora")

dst.mkdir(parents=True, exist_ok=True)

for p in list(src1.glob("*.jpg")) + list(src2.glob("*.jpg")):
    target = dst / p.name
    if not target.exists():
        shutil.copy2(p, target)

len(list(dst.glob("*.jpg")))

In [ ]:
meta_path = dst / "metadata.jsonl"

rows = 0
with open(meta_path, "w") as f:
    for _, row in data.iterrows():
        image_id = row["image_id"]
        if not image_id.endswith(".jpg"):
            image_id += ".jpg"

        img_path = dst / image_id
        if not img_path.exists():
            continue

        prompt = row["Prompt"]

        f.write(json.dumps({"file_name": image_id, "text": prompt}) + "\n")
        rows += 1

print("Wrote", rows, "rows to metadata.jsonl")

## Inference

In [ ]:
pipeline_base = AutoPipelineForText2Image.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
).to(device)

prompt = "Dermoscopy image of a melanoma on the upper back of a 65-year-old male."

out = pipeline_base(prompt, num_inference_steps=30, guidance_scale=7.5)
image_base = out.images[0]

In [ ]:
pipeline = AutoPipelineForText2Image.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
).to(device)

pipeline.load_lora_weights(
    "LoRA Weights",
    weight_name="pytorch_lora_weights.safetensors"
)

generated_prompt = "Dermoscopy image of a melanoma on the upper back of a 65-year-old male."
image = pipeline(prompt, num_inference_steps=30, guidance_scale=7.5).images[0]
image.save('./data/generated.png')

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(8,4))
axes[0].imshow(image)
axes[0].set_title('After Fine-Tuning')
axes[0].axis('off')
axes[1].imshow(image_base)
axes[1].set_title('Before Fine-Tuning')
axes[1].axis('off')
plt.suptitle('Sampling Comparison', fontweight = 'bold', fontsize=14)
plt.tight_layout()
plt.show()

## Analyzing Result

In [ ]:
def get_clip_embedding(img, processor, model):
    inputs = processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        emb = model.get_image_features(**inputs)
    emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.cpu()


def compute_ssim(img1, img2, size=(256, 256)):
    """Compute SSIM between two images resized to `size`."""
    img1_arr = np.array(img1.resize(size))
    img2_arr = np.array(img2.resize(size))
    s, _ = ssim(img1_arr, img2_arr, channel_axis=2, full=True)
    return float(s)

In [ ]:
generated_image_path = Path("./data/generated.png")
TOP_K = 5

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

gen_img = Image.open(generated_image_path).convert("RGB")
gen_emb = get_clip_embedding(gen_img, processor, model)

embeddings = []
paths = []

for p in tqdm(list(dst.rglob("*.jpg")), desc="Embedding dataset"):
    try:
        img = Image.open(p).convert("RGB")
    except:
        continue
    emb = get_clip_embedding(img, processor, model)
    embeddings.append(emb)
    paths.append(p)

embeddings = torch.cat(embeddings, dim=0)
similarities = (embeddings @ gen_emb.T).squeeze(1)

topk = torch.topk(similarities, k=TOP_K)
top_scores = topk.values.tolist()
top_indices = topk.indices.tolist()

top_k_paths = []
top_k_clip = []
top_k_ssim = []

for score, idx in zip(top_scores, top_indices):
    candidate_path = paths[idx]
    cand_img = Image.open(candidate_path).convert("RGB")
    ssim_score = compute_ssim(gen_img, cand_img)
    top_k_paths.append(candidate_path)
    top_k_clip.append(score)
    top_k_ssim.append(ssim_score)

In [ ]:
def get_prompt_for_path(path, metadata):
    fname = os.path.basename(str(path))
    return metadata.get(fname, "(no prompt found)")

In [ ]:
metadata = {}
with open(meta_path, "r") as f:
    for line in f:
        entry = json.loads(line.strip())
        metadata[entry["file_name"]] = entry["text"]

data_rows = [{
    "path": "GENERATED",
    "clip_score": '-',
    "ssim_score": '-',
    "prompt": generated_prompt
}]

for path, clip, ssim in zip(top_k_paths, top_k_clip, top_k_ssim):
    data_rows.append({
        "path": str(path),
        "clip_score": clip,
        "ssim_score": ssim,
        "prompt": metadata.get(Path(path).name, "NOT FOUND")
    })

df = pd.DataFrame(data_rows)
print(df)

similar_imgs = [Image.open(p).convert("RGB") for p in top_k_paths]
n_cols = 1 + len(similar_imgs)

fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 6))

axes[0].imshow(gen_img)
axes[0].set_title("Generated")
axes[0].axis("off")

for i, (img, clip_score, ssim_score) in enumerate(
    zip(similar_imgs, top_k_clip, top_k_ssim), start=1
):
    ax = axes[i]
    ax.imshow(img)
    ax.set_title(f"Similar #{i}\nCLIP={clip_score:.3f}\nSSIM={ssim_score:.3f}")
    ax.axis("off")

plt.subplots_adjust(bottom=0.1, wspace=0.3)
plt.show()